# LeetCode 3700 — Number of ZigZag Arrays II bằng Python

- Link: https://leetcode.com/problems/number-of-zigzag-arrays-ii/
- Độ khó: **Hard**
- Chủ đề: DP tuyến tính → **luỹ thừa ma trận** (fast matrix exponentiation), **đối xứng** để giảm kích thước ma trận, modulo `10^9 + 7`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
```

## 1. Mục tiêu học

1. Nhắc lại DP của bản I: `U/D` theo (giá trị cuối, hướng bước cuối) — mỗi bước là một **phép biến đổi tuyến tính**.
2. Viết phép biến đổi thành **ma trận** và tính `T^{n−2}` bằng **bình phương liên tiếp** trong `O(k³ log n)`.
3. Dùng đối xứng `D[v] = U[m−1−v]` để giảm ma trận từ `2m × 2m` xuống `m × m` — nhanh gấp 8 lần.
4. Cài phép nhân ma trận modulo đủ nhanh trong Python (`zip` + chuyển vị).
5. Kiểm chứng với DP tuyến tính (bản I) và vét cạn.

## 2. Đề bài & ví dụ

Giống bản I: dãy độ dài `n`, phần tử trong `[l, r]`, hai phần tử kề khác nhau, không có ba phần tử liên tiếp đơn điệu ngặt. Đếm modulo `10^9 + 7`.

**Ví dụ 1:** `n = 3, l = 4, r = 5` → `2`.

**Ví dụ 2:** `n = 3, l = 1, r = 3` → `10`.

**Ràng buộc (khác bản I):** `3 ≤ n ≤ 10^9`, `1 ≤ l < r ≤ 75`.

In [1]:
from itertools import product, accumulate
import random, time

MOD = 10**9 + 7
examples = [(3, 4, 5, 2), (3, 1, 3, 10)]
for n, l, r, exp in examples:
    print(n, l, r, "-> kỳ vọng", exp)

3 4 5 -> kỳ vọng 2
3 1 3 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Nhắc lại DP của bản I

Với `m = r − l + 1`, `U_i[v]` = số dãy độ dài `i` kết thúc tại `v` bằng bước lên. Nhờ đối xứng:

`U_{i+1}[v] = Σ_{w ≥ m−v} U_i[w]`, `U_2[v] = v`, đáp án `2·Σ U_n`.

Bản I chạy `n − 2` bước, mỗi bước `O(m)`. Với `n = 10^9` thì quá chậm.

In [2]:
def dp_linear(n, l, r):
    m = r - l + 1; U = list(range(m))
    for _ in range(n - 2):
        suf = list(accumulate(reversed(U)))[::-1] + [0]
        U = [suf[m - v] % MOD for v in range(m)]
    return 2 * sum(U) % MOD

print([dp_linear(n, l, r) for n, l, r, _ in examples])

[2, 10]


### 3.2. Luỹ thừa nhanh

`T^e`: nếu `e` chẵn, `T^e = (T^{e/2})²`; nếu lẻ, nhân thêm `T`. Duyệt bit của `e` ⇒ `O(log e)` phép nhân ma trận. Mỗi phép nhân `k × k` tốn `O(k³)`.

In [3]:
def mat_mul(A, B):
    Bt = list(zip(*B))
    return [[sum(a * b for a, b in zip(row, col)) % MOD for col in Bt] for row in A]

def mat_pow(T, e):
    k = len(T)
    R = [[int(i == j) for j in range(k)] for i in range(k)]
    while e:
        if e & 1: R = mat_mul(R, T)
        T = mat_mul(T, T); e >>= 1
    return R

F = [[1, 1], [1, 0]]
print("Fibonacci qua ma trận: F(50) =", mat_pow(F, 50)[0][1])

Fibonacci qua ma trận: F(50) = 586268941


### 3.3. Ma trận chuyển trạng thái

`U' = T·U` với `T[v][w] = 1` khi `w ≥ m − v`, ngược lại 0. Hàng `v` có `v` số 1 nằm ở cuối hàng.

In [4]:
def transition(m):
    return [[1 if w >= m - v else 0 for w in range(m)] for v in range(m)]

for row in transition(4): print(row)

[0, 0, 0, 0]
[0, 0, 0, 1]
[0, 0, 1, 1]
[0, 1, 1, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`U_n = T^{n−2} · U_2` ⇒ đáp án `= 2 · Σ_v (T^{n−2} U_2)[v]`. Chi phí: `log₂(10^9) ≈ 30` lần bình phương + tối đa 30 lần nhân, mỗi lần `m³ = 75³ ≈ 4,2·10^5` phép nhân-cộng ⇒ khoảng `2,5·10^7` phép — chạy được trong Python với cài đặt dùng `zip`.

Không dùng đối xứng thì ma trận `2m × 2m = 150 × 150`, chi phí gấp `2³ = 8` lần.

So với bản I: ở đó `m` tới 2000 nhưng `n ≤ 2000` (DP tuyến tính `O(n·m)` là đủ); ở bản II ngược lại, `n` khổng lồ nhưng `m ≤ 75` — đúng "vùng" của luỹ thừa ma trận. Đọc ràng buộc là cách nhanh nhất để chọn kỹ thuật.

## 5. Cách 1 — Brute Force

In [5]:
def is_zigzag(a):
    if any(a[i] == a[i + 1] for i in range(len(a) - 1)): return False
    return not any((a[i] < a[i + 1] < a[i + 2]) or (a[i] > a[i + 1] > a[i + 2]) for i in range(len(a) - 2))

def brute(n, l, r):
    return sum(1 for a in product(range(l, r + 1), repeat=n) if is_zigzag(a)) % MOD

for n, l, r, exp in examples:
    print(brute(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


**Nút thắt:** `m^n` — và ngay cả DP tuyến tính của bản I cũng cần `n · m = 7,5·10^{10}` phép với cận của bản II.

## 6. Ý tưởng thuật toán / bước đột phá

Bất kỳ DP nào có dạng `state_{i+1} = M · state_i` (hệ số không phụ thuộc `i`) đều tính được `state_n` trong `O(k³ log n)`. Ở đây `k = m ≤ 75` sau khi rút gọn đối xứng.

Mẹo nhỏ: thay vì tính `T^{n−2}` rồi nhân vector, có thể nhân vector **trực tiếp** khi bit tương ứng bật (`v ← P·v`), chỉ cần bình phương `P` — giảm một nửa số phép nhân ma trận–ma trận.

## 7. Các cách tiếp theo

- **Cách 2:** DP tuyến tính của bản I — `O(n·m)` (chỉ dùng cho `n` nhỏ, để kiểm chứng).
- **Cách 3:** luỹ thừa ma trận đầy đủ `2m × 2m` trên `(U, D)` — `O((2m)³ log n)`.
- **Cách 4:** luỹ thừa ma trận rút gọn `m × m` + nhân vector khi bit bật (lời giải nộp).

### 7.1. Cách 2 — DP tuyến tính (bản I)

Đã có ở mục 3.1 (`dp_linear`). Với `n` cỡ vài nghìn vẫn nhanh, dùng làm "đáp án chuẩn" cho kiểm thử.

### 7.2. Cách 3 — Ma trận `2m × 2m`

In [6]:
def full_matrix(n, l, r):
    m = r - l + 1; k = 2 * m
    M = [[0] * k for _ in range(k)]
    for v in range(m):
        for u in range(v):            M[v][m + u] = 1      # U'[v] += D[u], u < v
        for u in range(v + 1, m):     M[m + v][u] = 1      # D'[v] += U[u], u > v
    vec = list(range(m)) + [m - 1 - v for v in range(m)]
    P = mat_pow(M, n - 2)
    return sum(sum(P[i][j] * vec[j] for j in range(k)) for i in range(k)) % MOD

for n, l, r, exp in examples:
    print(full_matrix(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


### 7.3. Cách 4 — Ma trận `m × m` + nhân vector (lời giải chính)

In [7]:
def mat_vec(A, v):
    return [sum(a * b for a, b in zip(row, v)) % MOD for row in A]

def reduced_matrix(n, l, r):
    m = r - l + 1
    P = transition(m)
    U = list(range(m))
    e = n - 2
    while e:
        if e & 1: U = mat_vec(P, U)            # P = T^(2^j): các luỹ thừa giao hoán nên thứ tự không quan trọng
        e >>= 1
        if e: P = mat_mul(P, P)
    return 2 * sum(U) % MOD

for n, l, r, exp in examples:
    print(reduced_matrix(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


## 8. TRACE từng bước

In [8]:
def trace(n, l, r):
    m = r - l + 1; P = transition(m); U = list(range(m)); e = n - 2; j = 0
    print(f"m = {m}, cần T^{e}, e nhị phân = {bin(e)}; U_2 = {U}")
    while e:
        if e & 1:
            U = mat_vec(P, U); print(f"  bit {j} bật: U <- T^{2**j}·U = {U}")
        else:
            print(f"  bit {j} tắt")
        e >>= 1; j += 1
        if e: P = mat_mul(P, P)
    print("đáp án:", 2 * sum(U) % MOD, "| DP tuyến tính:", dp_linear(n, l, r))

trace(13, 1, 4)

m = 4, cần T^11, e nhị phân = 0b1011; U_2 = [0, 1, 2, 3]
  bit 0 bật: U <- T^1·U = [0, 3, 5, 6]
  bit 1 bật: U <- T^2·U = [0, 14, 25, 31]
  bit 2 tắt
  bit 3 bật: U <- T^8·U = [0, 8997, 16212, 20216]
đáp án: 90850 | DP tuyến tính: 90850


In [9]:
# Tăng trưởng: tỉ số đáp án liên tiếp tiến tới trị riêng lớn nhất của T
m = 5; vals = [dp_linear(n, 1, m) for n in range(3, 16)]
print([round(vals[i + 1] / vals[i], 4) for i in range(len(vals) - 1)])

[2.8333, 2.8941, 2.874, 2.8812, 2.8787, 2.8796, 2.8793, 2.8794, 2.8794, 2.8794, 2.8794, 2.8794]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
        MOD = 10**9 + 7
        m = r - l + 1

        def mul(A, B):
            Bt = list(zip(*B))
            return [[sum(a * b for a, b in zip(row, col)) % MOD for col in Bt] for row in A]

        P = [[1 if w >= m - v else 0 for w in range(m)] for v in range(m)]
        U = list(range(m))
        e = n - 2
        while e:
            if e & 1:
                U = [sum(a * b for a, b in zip(row, U)) % MOD for row in P]
            e >>= 1
            if e:
                P = mul(P, P)
        return 2 * sum(U) % MOD

sol = Solution()
for n, l, r, exp in examples:
    got = sol.zigZagArrays(n, l, r)
    print(n, l, r, "->", got, "✔" if got == exp else "✘")

3 4 5 -> 2 ✔
3 1 3 -> 10 ✔


## 10. Vì sao thuật toán đúng

1. **DP đúng** (xem bản I): `U_{i+1} = T·U_i`, `U_2[v] = v`, đáp án `2 Σ U_n` nhờ đối xứng phản chiếu.
2. **Luỹ thừa:** `U_n = T^{n−2} U_2`. Viết `n − 2 = Σ_j b_j 2^j`. Vòng lặp duy trì `P = T^{2^j}` và `U = T^{(bit 0..j−1 đã xử lý)} U_2`. Vì mọi luỹ thừa của cùng `T` giao hoán, áp `P` lên `U` khi `b_j = 1` cho `U = T^{Σ_{bit đã xử lý}} U_2`. Kết thúc: `U = T^{n−2} U_2`.
3. **Ma trận đầy đủ (Cách 3)** biểu diễn cùng hệ `U' = A·D`, `D' = B·U` mà không dùng đối xứng — kết quả trùng, dùng để kiểm chứng.
4. Mọi phép tính lấy modulo `10^9 + 7` (đồng cấu vành) nên kết quả cuối đúng modulo.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m^n)` | `O(n)` |
| 2. DP tuyến tính | `O(n · m)` | `O(m)` |
| 3. Ma trận `2m` | `O((2m)³ log n)` | `O(m²)` |
| 4. Ma trận `m` + vector | `O(m³ log n)` | `O(m²)` |

Với `m = 75`, `log n = 30`: Cách 4 ≈ `30 · 4,2·10^5 ≈ 1,3·10^7` phép nhân-cộng.

## 12. Bộ test

In [11]:
METHODS = [brute, dp_linear, full_matrix, reduced_matrix, lambda n, l, r: Solution().zigZagArrays(n, l, r)]
for n, l, r, exp in examples:
    assert all(f(n, l, r) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 lưới nhỏ so với vét cạn
for n in range(3, 7):
    for l in range(1, 3):
        for r in range(l + 1, 6):
            b = brute(n, l, r)
            assert all(f(n, l, r) == b for f in METHODS[1:]), (n, l, r)
assert Solution().zigZagArrays(10**9, 1, 2) == 2              # m = 2: luôn đúng 2 dãy
print("Lưới nhỏ: OK")

Lưới nhỏ: OK


In [13]:
# 12.3 n vừa: so với DP tuyến tính
rng = random.Random(3700)
for _ in range(60):
    n = rng.randint(3, 400); l = rng.randint(1, 40); r = rng.randint(l + 1, min(75, l + 30))
    d = dp_linear(n, l, r)
    assert reduced_matrix(n, l, r) == d == Solution().zigZagArrays(n, l, r), (n, l, r)
    if r - l < 12: assert full_matrix(n, l, r) == d
print("60 test n vừa: OK")

60 test n vừa: OK


In [14]:
# 12.4 cận lớn nhất
t0 = time.time(); x = Solution().zigZagArrays(10**9, 1, 75); t1 = time.time()
print("n = 10^9, m = 75:", x, "| thời gian:", round(t1 - t0, 2), "s")
t0 = time.time(); y = full_matrix(10**6, 1, 20); t1 = time.time()
print("đối chiếu ma trận đầy đủ (m = 20, n = 10^6):", y == Solution().zigZagArrays(10**6, 1, 20), "| thời gian:", round(t1 - t0, 2), "s")

n = 10^9, m = 75: 16 | thời gian: 0.75 s
đối chiếu ma trận đầy đủ (m = 20, n = 10^6): True | thời gian: 0.1 s


**Ghi chú kiểm chứng.** Đáp án `16` cho `n = 10^9, m = 75` trông "tròn" bất thường, nên đã được đối chiếu thêm bằng ma trận đầy đủ `150 × 150` (Cách 3) khi soạn notebook — hai cách độc lập cho cùng kết quả (Cách 3 mất khoảng 10 giây nên không chạy lại ở đây). Đáp án nhỏ chỉ là trùng hợp của phép lấy modulo.

**Vì sao đáng tin?** Hai cách dựa trên hai mô hình khác nhau: Cách 3 không dùng đối xứng, Cách 4 dùng. Nếu lập luận đối xứng sai, hai kết quả sẽ lệch nhau ngay với `n` nhỏ (mục 12.3) — và chúng khớp trên mọi ca thử.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luỹ thừa sai số mũ (`n − 1` hoặc `n` thay vì `n − 2`)

Vector khởi tạo là trạng thái ở độ dài **2**.

In [15]:
def wrong_exponent(n, l, r):
    m = r - l + 1; P = transition(m); U = list(range(m)); e = n - 1           # SAI
    while e:
        if e & 1: U = mat_vec(P, U)
        e >>= 1
        if e: P = mat_mul(P, P)
    return 2 * sum(U) % MOD

print("sai:", wrong_exponent(3, 1, 3), "| đúng:", brute(3, 1, 3))

sai: 16 | đúng: 10


### Lỗi 2 — Quên `% MOD` trong phép nhân ma trận

Số nguyên Python không tràn nhưng các phần tử phình to tới hàng trăm triệu chữ số ⇒ cực chậm (và ở C++/Java thì tràn số). Ô dưới đo kích thước sau vài lần bình phương **không** modulo:

In [16]:
def mul_no_mod(A, B):
    Bt = list(zip(*B)); return [[sum(a * b for a, b in zip(row, col)) for col in Bt] for row in A]

P = transition(30)
for _ in range(8): P = mul_no_mod(P, P)
print("số chữ số của phần tử lớn nhất sau 8 lần bình phương:", len(str(max(max(r) for r in P))))

số chữ số của phần tử lớn nhất sau 8 lần bình phương: 325


### Lỗi 3 — Xây ma trận đối xứng sai chỉ số (`w > m − v`)

In [17]:
def wrong_matrix(n, l, r):
    m = r - l + 1; P = [[1 if w > m - v else 0 for w in range(m)] for v in range(m)]   # SAI: lệch 1
    U = list(range(m)); e = n - 2
    while e:
        if e & 1: U = mat_vec(P, U)
        e >>= 1
        if e: P = mat_mul(P, P)
    return 2 * sum(U) % MOD

print("sai:", wrong_matrix(4, 1, 4), "| đúng:", brute(4, 1, 4))

sai: 26 | đúng: 62


### Lỗi 4 — Nhân vector theo thứ tự sai với ma trận **không** giao hoán

Mẹo "nhân vector khi bit bật" chỉ đúng vì mọi `P` là luỹ thừa của **cùng một** `T`. Nếu các bước dùng ma trận khác nhau (ví dụ xen kẽ hai ma trận `A`, `B` cho bước lẻ/chẵn), phải giữ đúng thứ tự `… B·A·B·A·v`.

In [18]:
A = [[1, 1], [0, 1]]; B = [[1, 0], [1, 1]]
AB = mat_mul(A, B); BA = mat_mul(B, A)
print("A·B =", AB, "| B·A =", BA, "| giao hoán?", AB == BA)

A·B = [[2, 1], [1, 1]] | B·A = [[1, 1], [1, 2]] | giao hoán? False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n` khổng lồ (`10^9`, `10^18`) + DP có số trạng thái nhỏ và chuyển trạng thái **không phụ thuộc bước** ⇒ luỹ thừa ma trận.
- Rút gọn trạng thái (đối xứng, gộp) trước khi luỹ thừa: chi phí tỉ lệ **lập phương** số trạng thái.
- Trong Python: chuyển vị `B` một lần (`zip(*B)`) rồi dùng `sum(a*b for a,b in zip(row, col))`.
- Chỉ cần vector kết quả ⇒ nhân ma trận–vector khi bit bật, giảm một nửa số phép nhân ma trận–ma trận.
- Có thể kiểm tra cài đặt luỹ thừa ma trận trên một bài đã biết đáp án (ví dụ Fibonacci ở mục 3.2) trước khi áp vào bài chính.
- Nếu dãy đáp án theo `n` thoả hồi quy tuyến tính bậc nhỏ (Bài 5: bậc 3 khi `m = 4`), Berlekamp–Massey + nhân đa thức còn nhanh hơn luỹ thừa ma trận.

## 15. Biến thể của bài

1. **Bản I** (`n ≤ 2000`, `m ≤ 2000`): DP tuyến tính `O(n m)`.
2. **Tổng theo mọi độ dài `3..n`**: thêm một trạng thái "tích luỹ" vào ma trận (ma trận khối `[[T, 0], [1ᵀ, 1]]`).
3. **Đếm dãy kết thúc tại giá trị `v` cố định**: đọc `U_n[v] + U_n[m−1−v]`.
4. **Thay luỹ thừa ma trận bằng Berlekamp–Massey**: dãy đáp án thoả một hồi quy tuyến tính bậc `≤ m`.

In [19]:
# Biến thể 2: tổng số dãy ZigZag với mọi độ dài 3..n, dùng ma trận khối (m+1) x (m+1)
def total_up_to(n, l, r):
    m = r - l + 1; T = transition(m)
    # trạng thái [U (m phần tử), S]: S' = S + Σ U'  (cộng đáp án của độ dài mới / 2)
    k = m + 1
    M = [[0] * k for _ in range(k)]
    for v in range(m):
        for w in range(m): M[v][w] = T[v][w]
    for w in range(m):
        M[m][w] = sum(T[v][w] for v in range(m))        # Σ_v (T·U)[v]
    M[m][m] = 1
    vec = list(range(m)) + [0]
    P = mat_pow(M, n - 2)
    res = [sum(P[i][j] * vec[j] for j in range(k)) % MOD for i in range(k)]
    return 2 * res[m] % MOD

for n in range(3, 9):
    assert total_up_to(n, 1, 4) == sum(dp_linear(t, 1, 4) for t in range(3, n + 1)) % MOD
print("Biến thể 2 OK; tổng độ dài 3..10^9 với m = 10:", total_up_to(10**9, 1, 10))

Biến thể 2 OK; tổng độ dài 3..10^9 với m = 10: 999999903


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi dãy | `O(m^n)` | Kiểm chứng |
| 2. DP tuyến tính | Bản I | `O(n m)` | `n` nhỏ |
| 3. Ma trận `2m` | Không rút gọn | `O(8 m³ log n)` | Kiểm chứng |
| 4. Ma trận `m` + vector | Đối xứng + luỹ thừa nhanh | `O(m³ log n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `zigZagArrays(10^9, 7, 8)`.

**Bài 2 (tính tay).** Viết ma trận `T` cho `m = 3` và tính `T·U_2`.

**Bài 3 (điền TODO).** Viết `count_ending_at(n, l, r, x)` — số dãy độ dài `n` kết thúc bằng giá trị `x`.

**Bài 4 (tự viết lại).** Viết lại lời giải bằng cách tính `T^{n−2}` đầy đủ rồi mới nhân vector; so thời gian.

**Bài 5 (mini challenge).** Tìm hồi quy tuyến tính của dãy `a_n = zigZagArrays(n, 1, 4)` bằng Berlekamp–Massey (mod `10^9 + 7`) và kiểm tra nó dự đoán đúng các số hạng sau.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_ending_at(n, l, r, x):
    # TODO: U_n[v] + U_n[m-1-v] với v = x - l
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def berlekamp_massey(seq):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `m = 2`. (2) Dãy buộc luân phiên. (3) `2`.

**Bài 2.** (1) `T = [[0,0,0],[0,0,1],[0,1,1]]`. (2) `U_2 = [0,1,2]`. (3) `T·U_2 = [0, 2, 3]` ⇒ `n = 3` có `2·5 = 10` dãy.

**Bài 3.** (1) Tính `U_n` như lời giải. (2) Số dãy kết thúc tại `v` bằng bước xuống là `U_n[m−1−v]`. (3) Cộng hai phần.

**Bài 4.** (1) `R = T^{n−2}` bằng `mat_pow`. (2) `U_n = R·U_2`. (3) Chậm hơn khoảng hai lần.

**Bài 5.** (1) Sinh ~`2m` số hạng đầu bằng DP. (2) BM cho đa thức đặc trưng. (3) Dự đoán số hạng tiếp theo và so với DP.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().zigZagArrays(10**9, 7, 8) == 2
T3 = transition(3)
assert T3 == [[0, 0, 0], [0, 0, 1], [0, 1, 1]] and mat_vec(T3, [0, 1, 2]) == [0, 2, 3]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def count_ending_at_sol(n, l, r, x):
    m = r - l + 1; P = transition(m); U = list(range(m)); e = n - 2
    while e:
        if e & 1: U = mat_vec(P, U)
        e >>= 1
        if e: P = mat_mul(P, P)
    v = x - l
    return (U[v] + U[m - 1 - v]) % MOD

def full_power_then_vector(n, l, r):
    m = r - l + 1; R = mat_pow(transition(m), n - 2)
    return 2 * sum(mat_vec(R, list(range(m)))) % MOD

for n in range(3, 7):
    for x in range(1, 5):
        assert count_ending_at_sol(n, 1, 4, x) == sum(1 for a in product(range(1, 5), repeat=n) if is_zigzag(a) and a[-1] == x)
t0 = time.time(); a1 = full_power_then_vector(10**9, 1, 60); t1 = time.time(); a2 = Solution().zigZagArrays(10**9, 1, 60); t2 = time.time()
assert a1 == a2
print(f"Bài 3, 4 OK | T^(n-2) đầy đủ {t1 - t0:.2f}s vs nhân vector {t2 - t1:.2f}s")

Bài 3, 4 OK | T^(n-2) đầy đủ 0.95s vs nhân vector 0.38s


In [24]:
# Bài 5: Berlekamp–Massey modulo nguyên tố
def berlekamp_massey_sol(s):
    C = [1]; B = [1]; L = 0; m = 1; b = 1
    for i in range(len(s)):
        d = s[i]
        for j in range(1, L + 1): d = (d + C[j] * s[i - j]) % MOD
        if d == 0: m += 1; continue
        coef = d * pow(b, MOD - 2, MOD) % MOD; T_ = C[:]
        C = C + [0] * (len(B) + m - len(C))
        for j in range(len(B)): C[j + m] = (C[j + m] - coef * B[j]) % MOD
        if 2 * L <= i: L = i + 1 - L; B = T_; b = d; m = 1
        else: m += 1
    return [(-c) % MOD for c in C[1:L + 1]]          # a_n = Σ rec[j] a_{n-1-j}

seq = [dp_linear(n, 1, 4) for n in range(3, 30)]
rec = berlekamp_massey_sol(seq[:16])
pred = seq[:len(rec)]
for i in range(len(rec), len(seq)):
    pred.append(sum(rec[j] * pred[i - 1 - j] for j in range(len(rec))) % MOD)
assert pred == seq
print("Bài 5 OK: bậc hồi quy =", len(rec), "| hệ số:", rec)

Bài 5 OK: bậc hồi quy = 3 | hệ số: [2, 1, 1000000006]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
m = r - l + 1
P = [[1 if w >= m - v else 0 for w in range(m)] for v in range(m)]   # T
U = list(range(m)); e = n - 2
while e:
    if e & 1: U = P · U          # ma trận–vector
    e >>= 1
    if e: P = P · P              # bình phương, % MOD
return 2 * sum(U) % MOD
```

**Checklist:**

- [ ] Số mũ `n − 2` (khởi tạo ở độ dài 2)?
- [ ] Ma trận `T[v][w] = [w ≥ m − v]`?
- [ ] `% MOD` trong mọi phép nhân?
- [ ] Rút gọn đối xứng để ma trận chỉ `m × m`?
- [ ] Đã đối chiếu với DP tuyến tính cho `n` vừa?
- [ ] Thời gian với `n = 10^9, m = 75` dưới vài giây?